# LLM Colors Preference Analysis
Analyze and compare results of adaptive preference elicitation (GRUM) from LLMs across models, embeddings, and sampling criteria.

In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import os

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

# Global Configuration
color_names = ["blue", "red", "green", "purple", "yellow"]
color_palette = {"blue": "#1f77b4", "red": "#d62728", "green": "#2ca02c", "purple": "#9467bd", "yellow": "#d6d627"}

## 1. Load Data
We load results from the latest run directory. Each result file contains metadata identifying the model, criterion, and embedding method.

In [2]:
# Set experiment directory
EXP_DIR = ROOT / "results" / "llm" / "llm_colors-20260326-114531"

results_data = []
output_dir = EXP_DIR / "outputs"

if output_dir.exists():
    for f in sorted(output_dir.glob("*.json")):
        with open(f, "r") as j:
            res = json.load(j)
            
            # Metadata extraction
            criterion = res.get("criterion", "social")
            embedding = res.get("agent_config", {}).get("params", {}).get("embedding_method", "")
            if not embedding:
                embedding = "HS" if "hidden_state" in f.name else "ST"
            else:
                embedding = "HS" if "hidden_state" in embedding else "ST"
            
            model_full = res.get("agent_config", {}).get("params", {}).get("model_id", "").split("/")[-1]
            model_type = "Instruct" if "Instruct" in model_full else "Pretrained"
            
            # Data extraction
            history = res.get("history", {})
            if not history: continue
            
            last_step = str(sorted(map(int, history.keys()))[-1])
            delta = history[last_step]["grum"]["delta"]
            
            for i, color in enumerate(color_names):
                results_data.append({
                    "Model Type": model_type,
                    "Model ID": model_full,
                    "Embedding": embedding,
                    "Criterion": criterion,
                    "Color": color,
                    "Score (Delta)": delta[i]
                })

df = pd.DataFrame(results_data)
print(f"Loaded {len(df) // len(color_names)} experiment runs.")
display(df.drop_duplicates(['Model ID', 'Embedding', 'Criterion'])[['Model ID', 'Embedding', 'Criterion']])

## 2. Comparative Analysis

### 2.1 Model Comparison: Pretrained vs Instruct
How does instruction tuning affect learned intrinsic preferences?

In [3]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Model Type", palette="viridis")
plt.title("Average Preference Scores: Pretrained vs Instruct Models")
plt.show()

### 2.2 Embedding Comparison: HS vs ST
Compare features extracted from LLM Hidden States (HS) vs Sentence Transformers (ST).

In [4]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Embedding", palette="Set2")
plt.title("Average Preference Scores: Hidden States (HS) vs Sentence Transformers (ST)")
plt.show()

### 2.3 Criterion Comparison
How do different sampling methodologies (Social, Personalized, Random) influence results?

In [5]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="Color", y="Score (Delta)", hue="Criterion", palette="muted")
plt.title("Average Preference Scores by Elicitation Criterion")
plt.show()

## 3. Global Perspective: Individual Run Details
A comprehensive view across all experimental dimensions.

In [6]:
g = sns.FacetGrid(df, col="Criterion", row="Model Type", hue="Embedding", palette="tab10", height=4, aspect=1.2, margin_titles=True)
g.map_dataframe(sns.barplot, x="Color", y="Score (Delta)", alpha=0.7, palette=color_palette, hue="Color", dodge=False)
g.add_legend(title="Color")
plt.subplots_adjust(top=0.9)
g.fig.suptitle("Detailed Intrinsic Preferences (Delta) Grid")
plt.show()

## 4. The Identifiability Problem: Unbounded Utility Drift

During initial runs without utility normalization, we observed extreme "utility drift". Specifically:
- **ST Embeddings**: Utilities for all alternatives dropped consistently, reaching approximately -40.
- **HS Embeddings**: Utilities became polarized (e.g., all personal weights positive, others negative).

### Theoretical Cause
This occurs because pairwise preference models are only identifiable up to an additive constant. Without an anchoring constraint (like sum-to-zero normalization), the utility values can drift indefinitely. Additionally, if the data lacks **Strong Connectivity** (Condition 1 in the GRUM paper), the solution may be unbounded.

### Visualizing the Drift
Below we load results from the `colors_unbounded` artifact to demonstrate this behavior.

In [ ]:
from grums.experiments.utils import load_experiment_results

# Path to merged unbounded data in artifacts
artifact_path = "/home/lotanamit/grum4llm/artifacts/llm/colors_unbounded"

unbounded_results = {}
for method in ['hs', 'st']:
    path = os.path.join(artifact_path, method)
    if os.path.exists(path):
        unbounded_results[method] = load_experiment_results(path)

if unbounded_results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    for i, (method, res) in enumerate(unbounded_results.items()):
        # Use a sample run from social criterion
        social_runs = [k for k in res.keys() if 'social' in k]
        if not social_runs: continue
        
        sample_key = social_runs[0]
        history = res[sample_key]['history']
        
        steps = sorted(map(int, history.keys()))
        deltas = [history[str(s)]['grum']['delta'] for s in steps]
        deltas = np.array(deltas)
        
        for j in range(deltas.shape[1]):
            label = color_names[j] if 'color_names' in globals() else f'Color {j}'
            axes[i].plot(steps, deltas[:, j], label=label, color=list(color_palette.values())[j])
        
        axes[i].set_title(f'Unbounded Utility Drift ({method.upper()})')
        axes[i].set_xlabel('Elicitation Step')
        axes[i].set_ylabel('Utility (Delta)')
        axes[i].legend(ncol=2)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()